In [1]:
import os
import csv
import glob
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType

# ==============================================================================
# 1. SPARK CON MEMORIA SUFICIENTE PARA 302 COLUMNAS
# ==============================================================================
spark = SparkSession.builder \
    .appName("Kelmarsh-TurbineData-EDA") \
    .master("local[4]") \
    .config("spark.driver.memory", "12g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# ==============================================================================
# 2. EXTRAER HEADER REAL (última línea comentada con '#')
# ==============================================================================
base_dir = os.path.dirname(os.getcwd())
bronze_dir = os.path.join(base_dir, "data", "bronze")

reference_file = glob.glob(
    os.path.join(bronze_dir, "Kelmarsh_SCADA_2021_*", "Turbine_Data_Kelmarsh_1_*.csv")
)[0]

print(f"📄 Archivo de referencia: {os.path.basename(reference_file)}")

header_line = None
with open(reference_file, "r", encoding="utf-8") as f:
    for line in f:
        stripped = line.strip()
        if stripped.startswith("#"):
            header_line = stripped
        else:
            break

header_clean = header_line.lstrip("#").strip()
column_names = [c.strip() for c in next(csv.reader([header_clean]))]

print(f"✅ Columnas en header: {len(column_names)}")
print(f"   [0]  → '{column_names[0]}'")
print(f"   [1]  → '{column_names[1]}'")
print(f"   [-1] → '{column_names[-1]}'")

# ==============================================================================
# 3. CARGA DE DATOS 2021-2024 (TURBINA 1)
# ==============================================================================
target_years = ["2021", "2022", "2023", "2024"]
selected_paths = [
    os.path.join(bronze_dir, f"Kelmarsh_SCADA_{year}_*", "Turbine_Data_Kelmarsh_1_*.csv")
    for year in target_years
]

turbine_raw = spark.read \
    .option("header", "False") \
    .option("inferSchema", "True") \
    .option("comment", "#") \
    .csv(selected_paths)

# ==============================================================================
# 4. RENOMBRAR COLUMNAS CON NOMBRES REALES
# ==============================================================================
n_csv = len(turbine_raw.columns)
n_hdr = len(column_names)

if n_csv == n_hdr:
    turbine_df = turbine_raw.toDF(*column_names)
    print(f"\n✅ Renombrado correcto: {n_csv} columnas")
else:
    print(f"\n⚠️  Mismatch: CSV={n_csv} cols vs Header={n_hdr} cols — usando nombres raw")
    turbine_df = turbine_raw

# ==============================================================================
# 5. CACHE Y VOLUMETRÍA
# ==============================================================================
turbine_df.cache()
total_records = turbine_df.count()   # Materializa el cache
total_cols    = len(turbine_df.columns)

print(f"\n🎯 Registros totales (Turbina 1 · 2021-2024): {total_records:,}")
print(f"📐 Número de columnas: {total_cols}")

# ==============================================================================
# 6. SCHEMA CON NOMBRES REALES
# ==============================================================================
print("\n📋 SCHEMA CON NOMBRES REALES:")
turbine_df.printSchema()

# ==============================================================================
# 7. PREVIEW
# ==============================================================================
print("\n👁️  PRIMERAS 3 FILAS:")
turbine_df.show(3, truncate=40)

# ==============================================================================
# 8. ESTADÍSTICAS DESCRIPTIVAS EN LOTES (evita OOM con 302 columnas)
# ==============================================================================
numeric_cols = [
    f.name for f in turbine_df.schema.fields
    if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))
]

BATCH_SIZE  = 20
total_batches = -(-len(numeric_cols) // BATCH_SIZE)   # ceil division
all_stats   = []

print(f"\n📊 ESTADÍSTICAS DESCRIPTIVAS — {len(numeric_cols)} columnas numéricas "
      f"(procesando en {total_batches} lotes de {BATCH_SIZE}):")

for i in range(0, len(numeric_cols), BATCH_SIZE):
    batch = numeric_cols[i : i + BATCH_SIZE]
    print(f"  📦 Lote {i//BATCH_SIZE + 1}/{total_batches}: cols {i}–{i+len(batch)-1}")

    alias_map = {name: f"c{j}" for j, name in enumerate(batch)}

    batch_df = turbine_df.select([
        F.col(f"`{name}`").alias(alias_map[name]) for name in batch
    ]).summary("count", "mean", "stddev", "min", "25%", "75%", "max")

    batch_pd = batch_df.toPandas()
    batch_pd.columns = ["summary"] + batch
    all_stats.append(batch_pd.set_index("summary").T)

stats_final = pd.concat(all_stats)
stats_final.index.name = "column"

print("\n" + stats_final.to_string())

# ==============================================================================
# 9. NULOS / NaN POR COLUMNA EN LOTES
# ==============================================================================
all_nulls = []
total_batches_all = -(-len(turbine_df.columns) // BATCH_SIZE)

print(f"\n🚨 NULOS / NaN — procesando {len(turbine_df.columns)} columnas "
      f"en {total_batches_all} lotes:")

for i in range(0, len(turbine_df.columns), BATCH_SIZE):
    batch_fields = turbine_df.schema.fields[i : i + BATCH_SIZE]
    print(f"  📦 Lote {i//BATCH_SIZE + 1}/{total_batches_all}: cols {i}–{i+len(batch_fields)-1}")

    null_exprs = []
    for f in batch_fields:
        escaped = f"`{f.name}`"
        if isinstance(f.dataType, (DoubleType, FloatType)):
            expr = F.count(F.when(
                F.col(escaped).isNull() | F.isnan(F.col(escaped)), 1
            )).alias(f.name)
        else:
            expr = F.count(F.when(
                F.col(escaped).isNull(), 1
            )).alias(f.name)
        null_exprs.append(expr)

    batch_nulls = turbine_df.select(null_exprs).toPandas().T.reset_index()
    batch_nulls.columns = ["column", "null_or_nan"]
    all_nulls.append(batch_nulls)

nulls_final = pd.concat(all_nulls, ignore_index=True)
nulls_final["pct"] = (nulls_final["null_or_nan"] / total_records * 100).round(2)
nulls_final = nulls_final.sort_values("pct", ascending=False)

with_nulls  = nulls_final[nulls_final["null_or_nan"] > 0]
clean_cols  = (nulls_final["null_or_nan"] == 0).sum()

print(f"\nColumnas con nulos/NaN : {len(with_nulls)} de {len(nulls_final)}")
print(f"Columnas completamente limpias: {clean_cols}\n")
print(with_nulls.to_string(index=False))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/29 22:50:27 WARN Utils: Your hostname, medion, resolves to a loopback address: 127.0.1.1; using 192.168.1.113 instead (on interface wlo1)
26/05/29 22:50:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 22:50:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/29 22:50:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


📄 Archivo de referencia: Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv
✅ Columnas en header: 303
   [0]  → 'Date and time'
   [1]  → 'Wind speed (m/s)'
   [-1] → 'Tower Acceleration Y, StdDev (mm/ss)'



✅ Renombrado correcto: 303 columnas


26/05/29 22:51:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/29 22:51:32 WARN MemoryStore: Not enough space to cache rdd_14_30 in memory! (computed 235.6 MiB so far)
26/05/29 22:51:32 WARN BlockManager: Persisting block rdd_14_30 to disk instead.
26/05/29 22:51:32 WARN MemoryStore: Not enough space to cache rdd_14_30 in memory! (computed 235.6 MiB so far)
26/05/29 22:51:33 WARN MemoryStore: Not enough space to cache rdd_14_32 in memory! (computed 133.3 MiB so far)
26/05/29 22:51:33 WARN BlockManager: Persisting block rdd_14_32 to disk instead.
26/05/29 22:51:33 WARN MemoryStore: Not enough space to cache rdd_14_33 in memory! (computed 133.3 MiB so far)
26/05/29 22:51:33 WARN BlockManager: Persisting block rdd_14_33 to disk instead.
26/05/29 22:51:34 WARN MemoryStore: Not enough space to cache rdd_14_34 in memory! (computed 133.3 MiB so far)
26/05/29 22:5


🎯 Registros totales (Turbina 1 · 2021-2024): 4,454,784
📐 Número de columnas: 303

📋 SCHEMA CON NOMBRES REALES:
root
 |-- Date and time: timestamp (nullable = true)
 |-- Wind speed (m/s): double (nullable = true)
 |-- Wind speed, Standard deviation (m/s): double (nullable = true)
 |-- Wind speed, Minimum (m/s): double (nullable = true)
 |-- Wind speed, Maximum (m/s): double (nullable = true)
 |-- Long Term Wind (m/s): double (nullable = true)
 |-- Wind speed Sensor 1 (m/s): double (nullable = true)
 |-- Wind speed Sensor 1, Standard deviation (m/s): double (nullable = true)
 |-- Wind speed Sensor 1, Minimum (m/s): double (nullable = true)
 |-- Wind speed Sensor 1, Maximum (m/s): double (nullable = true)
 |-- Wind speed Sensor 2 (m/s): double (nullable = true)
 |-- Wind speed Sensor 2, Standard deviation (m/s): double (nullable = true)
 |-- Wind speed Sensor 2, Minimum (m/s): double (nullable = true)
 |-- Wind speed Sensor 2, Maximum (m/s): double (nullable = true)
 |-- Density adjusted

...

26/05/29 22:54:46 WARN MemoryStore: Not enough space to cache rdd_14_33 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:46 WARN MemoryStore: Not enough space to cache rdd_14_34 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:46 WARN MemoryStore: Not enough space to cache rdd_14_30 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:46 WARN MemoryStore: Not enough space to cache rdd_14_35 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:46 WARN MemoryStore: Not enough space to cache rdd_14_36 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:46 WARN MemoryStore: Not enough space to cache rdd_14_37 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:46 WARN MemoryStore: Not enough space to cache rdd_14_38 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_40 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_39 in memory! (computed 68.7 MiB so far)
26/05/29 2

  📦 Lote 16/16: cols 300–302


26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_33 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_34 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_30 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_35 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_37 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_36 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_38 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_39 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:47 WARN MemoryStore: Not enough space to cache rdd_14_40 in memory! (computed 68.7 MiB so far)
26/05/29 2


Columnas con nulos/NaN : 302 de 303
Columnas completamente limpias: 1

                                                  column  null_or_nan   pct
                             Energy Import counter (kWh)      4444858 99.78
                  Reactive Energy Import counter (kvarh)      4365168 97.99
                                       Performance Index      4353894 97.74
         Potential power primary reference turbines (kW)      4351084 97.67
                                Potential power MPC (kW)      4350517 97.66
                                   Power factor (cosphi)      4350724 97.66
                  Equivalent Full Load Hours counter (s)      4349733 97.64
            Potential power met mast anemometer MPC (kW)      4349733 97.64
                Potential power met mast anemometer (kW)      4349520 97.64
               Manufacturer Potential Power (SCADA) (kW)      4349520 97.64
                      Potential Power Energy Budget (kW)      4349664 97.64
                

26/05/29 22:54:48 WARN MemoryStore: Not enough space to cache rdd_14_47 in memory! (computed 42.0 MiB so far)
26/05/29 22:54:48 WARN MemoryStore: Not enough space to cache rdd_14_45 in memory! (computed 68.7 MiB so far)
26/05/29 22:54:48 WARN MemoryStore: Not enough space to cache rdd_14_46 in memory! (computed 44.7 MiB so far)
